In [1]:
import intake
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import geopandas as gpd

In [2]:
! pip install --upgrade xarray zarr gcsfs cftime nc-time-axis

In [3]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import gcsfs

xr.set_options(display_style='html')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [4]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
df['activity_id'].unique()

<StringArray>
[ 'HighResMIP',        'CMIP',       'CFMIP', 'ScenarioMIP',  'AerChemMIP',
       'RFMIP',      'FAFMIP',       'DAMIP',       'LUMIP',      'CDRMIP',
       'GMMIP',       'C4MIP',        'OMIP',        'PMIP',      'LS3MIP',
        'DCPP',       'PAMIP',      'ISMIP6']
Length: 18, dtype: str

In [5]:
df['variable_id'].unique()

<StringArray>
[               'ps',              'rsds',              'rlus',
              'rlds',               'psl',               'prw',
              'hurs',              'huss',               'hus',
              'hfss',
 ...
    'siforceintstrx',    'siforcecoriolx', 'ocontemppsmadvect',
          'difmxylo',       'msftyrhompa',       'msftmrhompa',
            'phabio',           'co3abio',          'fco2antt',
          'difmxybo']
Length: 709, dtype: str

In [6]:
hazard_vars = ['pr', 'prw', 'tas', 'mrro']

In [7]:
mrro_df = df[
    (df["activity_id"] == "CMIP") &
    (df["experiment_id"] == "historical") &
    (df["table_id"] == "Lmon") &
    (df["variable_id"]=='mrro') &
    (df["source_id"] == "GFDL-ESM4") &
    (df["member_id"] == "r1i1p1f1")
]

mrro_df

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
244819,CMIP,NOAA-GFDL,GFDL-ESM4,historical,r1i1p1f1,Lmon,mrro,gr1,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/hist...,NaN,20190726


In [8]:
pr_df = df[
    (df["activity_id"] == "CMIP") &
    (df["experiment_id"] == "historical") &
    (df["table_id"] == "Amon") &
    (df["variable_id"].isin(hazard_vars)) &
    (df["source_id"] == "GFDL-ESM4") &
    (df["member_id"] == "r1i1p1f1")
]

pr_df

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
244695,CMIP,NOAA-GFDL,GFDL-ESM4,historical,r1i1p1f1,Amon,tas,gr1,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/hist...,NaN,20190726
244713,CMIP,NOAA-GFDL,GFDL-ESM4,historical,r1i1p1f1,Amon,prw,gr1,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/hist...,NaN,20190726
244715,CMIP,NOAA-GFDL,GFDL-ESM4,historical,r1i1p1f1,Amon,pr,gr1,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/hist...,NaN,20190726


In [40]:
# this only needs to be created once
gcs = gcsfs.GCSFileSystem(token='anon')

# get the path to a specific zarr store (the first one from the dataframe above)
zstore = pr_df.loc[244695].zstore

# create a mutable-mapping-style interface to the store
mapper = gcs.get_mapper(zstore)

# open it using xarray and zarr
ds = xr.open_zarr(mapper, consolidated=True)
ds

<xarray.Dataset> Size: 411MB
Dimensions:    (time: 1980, lat: 180, lon: 288, bnds: 2)
Coordinates:
  * time       (time) object 16kB 1850-01-16 12:00:00 ... 2014-12-16 12:00:00
  * lat        (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon        (lon) float64 2kB 0.625 1.875 3.125 4.375 ... 356.9 358.1 359.4
  * bnds       (bnds) float64 16B 1.0 2.0
    lat_bnds   (lat, bnds) float64 3kB ...
    lon_bnds   (lon, bnds) float64 5kB ...
    time_bnds  (time, bnds) object 32kB ...
    height     float64 8B ...
Data variables:
    tas        (time, lat, lon) float32 411MB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.0 UGRID-1.0
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  36500.0
    comment:                <null ref>
    ...                     ...
    title:                  NOAA GFDL GFDL-ESM4 model output prepared for CMI...
    tracking_id:            hdl:21.14100/75e5c5a7-d7c4-4860-beb1-db454f25f13a...
    variable_id:            tas
    variant_info:           N/A
    variant_label:          r1i1p1f1
    status:                 2019-09-10;created;by nhn2@columbia.edu

In [41]:
df = ds.to_dataframe().reset_index()


In [42]:
month_names = {
    1: 'January',
    2: 'February',
    3: 'March',
    4: 'April',
    5: 'May',
    6: 'June',
    7: 'July',
    8: 'August',
    9: 'September',
    10: 'October',
    11: 'November',
    12: 'December'
}

df['month'] = df['time'].apply(lambda x: month_names[x.month])
df['year'] = df['time'].apply(lambda x: x.year)
df['day'] = df['time'].apply(lambda x: x.day)

In [43]:
import geopandas as gpd
import pandas as pd

# 1. Filter for the year 1999
df_year = df[(df['year'] == 2010)].copy()

# Fix unit conversion for precipitation ('pr') only (convert kg/m²/s to mm/day)
# If your 'pr' data is stored in a 'value' column alongside a 'variable_id', uncomment below:
# df_year.loc[df_year['variable_id'] == 'pr', 'value'] = df_year.loc[df_year['variable_id'] == 'pr', 'value'] * 86400

# Convert 0-360 longitude scale to standard -180 to 180 scale if needed
df_year['lon'] = df_year['lon'].apply(lambda x: x - 360 if x > 180 else x)

# 2. Map coordinates to spatial points
gdf = gpd.GeoDataFrame(
    df_year,
    geometry=gpd.points_from_xy(df_year["lon"], df_year["lat"]),
    crs="EPSG:4326"
)

# 3. Read from the 10m high-res global provinces/states dataset
provinces = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
)

# 4. Filter for US States requested (Texas, Louisiana, Pennsylvania)
target_states = provinces[
    (provinces["admin"] == "United States of America") & 
    (provinces["name"].isin([
        "Texas", "Louisiana", "Pennsylvania"
    ]))
]

print(f"Successfully loaded {len(target_states)} states for analysis.")

# 5. Spatial join: Matches climate grid points directly into the state boundaries
joined = gpd.sjoin(gdf, target_states, predicate="intersects")

# 6. Safety check
if len(joined) == 0:
    print("\n[ALERT] Spatial join resulted in 0 rows.")
    print("-> Check if your climate data longitude is scaled correctly or if the bounding boxes overlap.")
else:
    print(f"\nSuccess! Found {len(joined)} weather data records inside the target states.")

    # 7. Pivot/Group the data to get the MAX value per state per day
    # Note: If your rainfall data is in a column named 'value', change 'pr' below to 'value'
    state_hazards = joined.groupby(["name", "month", "day"]).agg(
        tas_max=('tas', 'max')
    ).reset_index()
    
    # 8. Export cleanly to your project storage
    output_filename = 'data/2010_tas_max.csv'
    state_hazards.to_csv(output_filename, index=False)
    print(f"Data export completed! Total rows saved: {len(state_hazards)}")
    print(f"Saved to: {output_filename}")

Successfully loaded 3 states for analysis.

Success! Found 1608 weather data records inside the target states.
Data export completed! Total rows saved: 36
Saved to: data/2010_tas_max.csv


In [39]:
import geopandas as gpd
import pandas as pd

# 1. Filter for the year 2010 (Pakistan Flood)
df_year = df[(df['year'] == 2010)].copy()

# Fix unit conversion for precipitation ('pr') only (convert kg/m²/s to mm/day)
# If your 'pr' data is stored in a 'value' column alongside a 'variable_id', uncomment below:
# df_year.loc[df_year['variable_id'] == 'pr', 'value'] = df_year.loc[df_year['variable_id'] == 'pr', 'value'] * 86400

# Convert 0-360 longitude scale to standard -180 to 180 scale if needed
df_year['lon'] = df_year['lon'].apply(lambda x: x - 360 if x > 180 else x)

# 2. Map coordinates to spatial points
gdf = gpd.GeoDataFrame(
    df_year,
    geometry=gpd.points_from_xy(df_year["lon"], df_year["lat"]),
    crs="EPSG:4326"
)

# 3. Read from the 10m high-res global provinces/states dataset
provinces = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
)

# 4. Filter for Pakistan and explicitly select your four target provinces
# Note: Natural Earth records Khyber Pakhtunkhwa as "Khyber Pakhtunkhwa"
target_provinces = provinces[
    (provinces["admin"] == "Pakistan") & 
    (provinces["name"].isin([
        "Khyber Pakhtunkhwa", "Punjab", "Sindh", "Balochistan"
    ]))
]

print(f"Successfully loaded {len(target_provinces)} provinces for analysis.")

# 5. Spatial join: Matches climate grid points directly into the provincial boundaries
joined = gpd.sjoin(gdf, target_provinces, predicate="intersects")

# 6. Safety check
if len(joined) == 0:
    print("\n[ALERT] Spatial join resulted in 0 rows.")
    print("-> Check if your climate data longitude is scaled correctly or if the bounding boxes overlap.")
else:
    print(f"\nSuccess! Found {len(joined)} weather data records inside the target provinces.")

    # 7. Pivot/Group the data to get the MAX value per province per day
    # Note: If your rainfall data is in a column named 'value', change 'pr' below to 'value'
    province_hazards = joined.groupby(["name", "month", "day"]).agg(
        property_max=('pr', 'max')
    ).reset_index()
    
    # 8. Export cleanly to your project storage
    output_filename = 'data/pakistan_provinces_2010_pr_max.csv'
    province_hazards.to_csv(output_filename, index=False)
    print(f"Data export completed! Total rows saved: {len(province_hazards)}")
    print(f"Saved to: {output_filename}")

Successfully loaded 1 provinces for analysis.

Success! Found 360 weather data records inside the target provinces.
Data export completed! Total rows saved: 12
Saved to: data/pakistan_provinces_2010_pr_max.csv


In [ ]:
import geopandas as gpd
import pandas as pd

df_year = df[(df['year'] == 1999)].copy()
# Fix unit conversion for precipitation ('pr') only (convert kg/m²/s to mm/day)
# Assuming your dataset stores values in a column named 'value'
# df_year['pr']=df_year['pr']* 86400
# df_year.loc[df_year['variable_id'] == 'pr', 'value'] = df_year.loc[df_year['variable_id'] == 'pr', 'value'] * 86400

# Convert 0-360 longitude scale to standard -180 to 180 scale if needed
df_year['lon'] = df_year['lon'].apply(lambda x: x - 360 if x > 180 else x)

# 2. Map coordinates to spatial points
gdf = gpd.GeoDataFrame(
    df_year,
    geometry=gpd.points_from_xy(df_year["lon"], df_year["lat"]),
    crs="EPSG:4326"
)

# 3. Read from the 10m high-res global provinces dataset
provinces = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
)

# 4. Filter for China provinces heavily impacted by the 1998 Yangtze River Flood
# These represent the core inland river basins that completely overflowed
yangtze_provinces = provinces[
    (provinces["admin"] == "China") & 
    (provinces["name"].isin([
        "Henan", "Hubei", "Shandong"
    ]))
]

print(f"Successfully loaded {len(yangtze_provinces)} provinces along the Yangtze River Basin.")

# 5. Spatial join: Matches climate grid points directly into Chinese province boundaries
joined = gpd.sjoin(gdf, yangtze_provinces, predicate="intersects")

# 6. Safety check
if len(joined) == 0:
    print("\n[ALERT] Spatial join resulted in 0 rows.")
    print("-> Check if your climate data longitude is scaled correctly or if the bounding boxes overlap.")
else:
    print(f"\nSuccess! Found {len(joined)} weather data records inside the target provinces.")

    # 7. Pivot/Group the data to get the MAX value for EACH hazard variable per province per day
    # Note: If your dataframe uses a column name other than 'value' for the measurements, update it here
    province_hazards = joined.groupby(["name", "month",'day']).agg(
        property_max=('pr', 'max')
    ).reset_index()
    # 8. Export cleanly to your project storage
    output_filename = 'data/max_1999_pr.csv'
    province_hazards.to_csv(output_filename, index=False)
    print(f"Data export completed! Total rows saved: {len(province_hazards)}")
    print(f"Saved to: {output_filename}")

Successfully loaded 3 provinces along the Yangtze River Basin.

Success! Found 912 weather data records inside the target provinces.
Data export completed! Total rows saved: 36
Saved to: data/max_1887_pr.csv


In [37]:
pr_1887 = pd.read_csv('data/max_1887_pr.csv')
tas_1887 = pd.read_csv('data/max_1887_tas.csv')
# pr_1998 = pd.read_csv('data/max_1998_pr.csv')
# tas_1998 = pd.read_csv('data/max_1998_tas.csv')
df_1887 = pr_1887.merge(tas_1887,on='name',how='outer')
df_1887 = df_1887.rename(columns={'month_x':'month','day_x':'day','property_max':'pr_max'})
df_1887=df_1887[['name','month','day','pr_max','tas_max']]
df_1887.to_csv('data/df_1887.csv')
# df_1998 = pr_1998.merge(tas_1998,on='name',how='outer')
# df_1998 = df_1998.rename(columns={'month_x':'month','day_x':'day'})
# df_1998=df_1998[['name','month','day','pr_max','tas_max']]
# df_1998.to_csv('data/df_1998.csv')




In [34]:
tas_1887.columns

Index(['name', 'month', 'day', 'tas_max'], dtype='str')

In [ ]:
import geopandas as gpd

# 1. Clean slice of your 2001 weather data (TS Allison occurred in June 2001)
df_1998 = df[df['year'] == 1998].copy()
df_1998['pr'] = df_1998['pr'] * 86400

# Convert 0-360 longitude scale to standard -180 to 180 scale if needed
df_1998['lon'] = df_1998['lon'].apply(lambda x: x - 360 if x > 180 else x)

# 2. Map coordinates to spatial points
gdf = gpd.GeoDataFrame(
    df_1998,
    geometry=gpd.points_from_xy(df_1998["lon"], df_1998["lat"]),
    crs="EPSG:4326"
)

# 3. Read from the 10m high-res cultural dataset
us_states = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
)

# 4. Filter for US states heavily impacted by Tropical Storm Allison's heavy rainfall track
# Epicenter was Texas/Louisiana, but the storm tracked all the way up the coast.
allison_states = us_states[
    (us_states["admin"] == "United States of America") & 
    (us_states["name"].isin(["Texas", "Louisiana", "Mississippi", "Alabama", "Florida", "Georgia", "North Carolina", "Virginia", "Maryland", "Pennsylvania"]))
]

print(f"Successfully loaded {len(allison_states)} regions for Tropical Storm Allison analysis.")

# 5. Spatial join: Matches weather points directly into US state polygons
joined = gpd.sjoin(gdf, allison_states, predicate="intersects")

# 6. Safety check: Ensure points actually matched the states
if len(joined) == 0:
    print("\n[ALERT] Spatial join resulted in 0 rows.")
    print(f"-> Your weather data bounding box: {gdf.total_bounds}")
    print(f"-> Target states bounding box: {allison_states.total_bounds}")
else:
    print(f"\nSuccess! Found {len(joined)} weather data records inside the target states.")

    # 7. Group metrics by state name, month, and day
    state_rain = joined.groupby(["name", "month", "day"]).agg({
        "pr": ["sum", "mean", "max"]
    }).reset_index()

    # 8. Flatten multi-level headers for clean charting scripts
    state_rain.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col 
        for col in state_rain.columns
    ]

    # 9. Export cleanly to target project storage
    state_rain.to_csv('data/ts_allison_2001_rain.csv', index=False)
    print(f"Data export completed! Total rows saved: {len(state_rain)}")

In [ ]:
df_1998 = df[df['year']==2012].copy()
df_2012['pr']=df_2012['pr']*86400

In [80]:
df.columns

Index(['time', 'lat', 'lon', 'bnds', 'pr', 'lat_bnds', 'lon_bnds', 'time_bnds',
       'month', 'year', 'day'],
      dtype='str')

In [42]:
def make_poly(row):
    return Polygon([
        (row["lon_bnds"][0], row["lat_bnds"][0]),
        (row["lon_bnds"][1], row["lat_bnds"][0]),
        (row["lon_bnds"][1], row["lat_bnds"][1]),
        (row["lon_bnds"][0], row["lat_bnds"][1])
    ])

In [60]:
gdf = gpd.GeoDataFrame(
    df_2017,
    geometry=gpd.points_from_xy(df_2017["lon"], df_2017["lat"]),
    crs="EPSG:4326"
)

In [62]:
us = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_1_states_provinces.zip"
)
us = us[us["admin"] == "United States of America"]

In [63]:
joined = gpd.sjoin(gdf, us, predicate="intersects")
province_rain = joined.groupby(["name", "month",'day']).agg({
    "pr": ["sum", "mean", "max"]
}).reset_index()
province_rain.to_csv('data/us.csv',index=False)
#province_rain.to_csv('data/province_pr.csv')


In [79]:
joined.columns

Index(['time', 'lat', 'lon', 'bnds', 'pr', 'lat_bnds', 'lon_bnds', 'time_bnds',
       'month', 'year',
       ...
       'FCLASS_TR', 'FCLASS_ID', 'FCLASS_PL', 'FCLASS_GR', 'FCLASS_IT',
       'FCLASS_NL', 'FCLASS_SE', 'FCLASS_BD', 'FCLASS_UA', 'FCLASS_TLC'],
      dtype='str', length=134)

In [77]:
import geopandas as gpd

# 1. Clean slice of your 2001 weather data (TS Allison occurred in June 2001)
df_2001 = df[df['year'] == 2001].copy()
df_2001['pr'] = df_2001['pr'] * 86400

# Convert 0-360 longitude scale to standard -180 to 180 scale if needed
df_2001['lon'] = df_2001['lon'].apply(lambda x: x - 360 if x > 180 else x)

# 2. Map coordinates to spatial points
gdf = gpd.GeoDataFrame(
    df_2001,
    geometry=gpd.points_from_xy(df_2001["lon"], df_2001["lat"]),
    crs="EPSG:4326"
)

# 3. Read from the 10m high-res cultural dataset
us_states = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
)

# 4. Filter for US states heavily impacted by Tropical Storm Allison's heavy rainfall track
# Epicenter was Texas/Louisiana, but the storm tracked all the way up the coast.
allison_states = us_states[
    (us_states["admin"] == "United States of America") & 
    (us_states["name"].isin(["Texas", "Louisiana", "Mississippi", "Alabama", "Florida", "Georgia", "North Carolina", "Virginia", "Maryland", "Pennsylvania"]))
]

print(f"Successfully loaded {len(allison_states)} regions for Tropical Storm Allison analysis.")

# 5. Spatial join: Matches weather points directly into US state polygons
joined = gpd.sjoin(gdf, allison_states, predicate="intersects")

# 6. Safety check: Ensure points actually matched the states
if len(joined) == 0:
    print("\n[ALERT] Spatial join resulted in 0 rows.")
    print(f"-> Your weather data bounding box: {gdf.total_bounds}")
    print(f"-> Target states bounding box: {allison_states.total_bounds}")
else:
    print(f"\nSuccess! Found {len(joined)} weather data records inside the target states.")

    # 7. Group metrics by state name, month, and day
    state_rain = joined.groupby(["name", "month", "day"]).agg({
        "pr": ["sum", "mean", "max"]
    }).reset_index()

    # 8. Flatten multi-level headers for clean charting scripts
    state_rain.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col 
        for col in state_rain.columns
    ]

    # 9. Export cleanly to target project storage
    state_rain.to_csv('data/ts_allison_2001_rain.csv', index=False)
    print(f"Data export completed! Total rows saved: {len(state_rain)}")

Successfully loaded 10 regions for Tropical Storm Allison analysis.

Success! Found 3096 weather data records inside the target states.
Data export completed! Total rows saved: 120


In [76]:
ve_states[(ve_states["admin"] == "Venezuela")&(ve_states["name"].isin(["Vargas", "La Guaira", "Distrito Capital", "Miranda"]))]


,featurecla,scalerank,adm1_code,diss_me,iso_3166_2,wikipedia,iso_a2,adm0_sr,name,name_alt,...,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,FCLASS_TLC,geometry
1897,Admin-1 states provinces,5,VEN-42,42,VE-X,NaN,VE,1,Vargas,NaN,...,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,"POLYGON ((-67.39175 10.54308, -67.32608 10.549..."
1898,Admin-1 states provinces,5,VEN-47,47,VE-M,NaN,VE,1,Miranda,NaN,...,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,"POLYGON ((-66.31361 10.62949, -66.30948 10.634..."
3216,Admin-1 states provinces,5,VEN-43,43,VE-A,NaN,VE,1,Distrito Capital,NaN,...,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,"POLYGON ((-67.00536 10.57519, -67.0042 10.5749..."


In [54]:
# Look for anything containing "United" or "America" to find the exact spelling
print(us_states[us_states['admin'].str.contains('United|America', na=False)]['admin'].unique())

<StringArray>
[         'United Republic of Tanzania',
                 'United Arab Emirates',
                       'United Kingdom',
             'United States of America',
 'United States Minor Outlying Islands',
         'United States Virgin Islands',
                       'American Samoa']
Length: 7, dtype: str


In [ ]:
df_filtered = joined[joined["name"].isin(["Hubei", "Hunan", "Jianxi",'Anhui','Jiangsu'])]

province_rain = df_filtered.groupby(["name", "month"]).agg({
    "pr": ["sum", "mean", "max"]
}).reset_index()

province_rain.columns = ["name", "month", "pr_sum", "pr_mean", "pr_max"]


# province_rain.columns = ["name", "pr_sum", "pr_mean", "pr_max"]
choropleth = china.merge(province_rain, on="name")
# choropleth.to_file(
#     "yangtze.geojson",
#     driver="GeoJSON"
# )

In [25]:
yangtze = choropleth[['name','month','pr_sum','pr_mean','pr_max']]
yangtze.to_csv('data/yangtze.csv',index=False)

In [26]:
yangtze['name'].unique()

<StringArray>
['Anhui', 'Hubei', 'Hunan', 'Jiangsu']
Length: 4, dtype: str

In [22]:
choro = pd.DataFrame(choropleth.drop(columns='geometry'))
choro.to_json('data/yangtze.json',orient='records',indent=4)